# 深度代理：从头开始构建研究代理

欢迎来到深度代理！本笔记本将通过从头开始**逐步构建研究代理**，引导您了解深度代理框架的核心概念。

**您将学到什么：**
- Deep Agents 是什么以及它开箱即用的功能
- 如何添加自定义工具来扩展代理功能
- 了解后端和存储抽象
- 具有子代理和上下文隔离的任务委托
- 人机交互模式以确保安全
- 持久存储的长期记忆
- 中间件架构和可扩展性
- 可重用代理能力的技能

<br>
<br>
---
<br>

> **注意：** 本研讨会需要 [Tavily API 密钥]( https://tavily.com) 来实现网络搜索功能。Deep Agents 构建在 LangGraph 之上，为构建具有文件系统访问、规划和委派功能的自主代理提供了强大的工具。

## 第 0 部分：设置和安装

首先，让我们安装必要的软件包并设置我们的环境。

In [ ]:
# 安装所需的包
# 运行 uvsync 来安装软件包或运行：
# !pip 安装 deepagents tavilly-python python-dotenv

zsh:1: command not found: pip


### 初始化你的LLM

In [2]:
# 将项目根添加到 Python 路径，以便我们可以从 utils 模块导入
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

    # 从集中式utils模块导入模型
from utils.models import model

# 为 Tavilly 加载环境变量
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env", override=True)

# 抑制警告以获得更干净的输出
import warnings

warnings.filterwarnings('ignore', message='LangSmith 现在使用 UUID v7')

## 第 1 部分：您的第一个深度代理（The Harness）

 <img src="../../images/deepAgentsDiag.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Deep Agents Architecture"> 

Deep Agents 充当**“代理工具”**——一个构建在核心工具调用循环上的框架，但具有用于自主任务执行的预构建工具和集成功能。

### 您开箱即用的内容：

- **文件系统工具** — `ls` 、 `read_file` 、 `write_file` 、 `edit_file` 、 `glob` 、 `grep` 
- **规划工具** — `write_todos` 用于任务跟踪
- **子代理委托** — 用于独立工作的 `task()` 工具
- **大型工具结果卸载** — 自动将工具结果 >20k 令牌卸载到文件系统
- **对话摘要** — 在接近约 85% 上下文容量时压缩历史记录
- **悬空工具调用修补** — 自动修复消息历史记录一致性

让我们创建最基本的深度代理：

In [3]:
from deepagents import create_deep_agent

# 创建最基本的深度代理——只是一个模型！
agent = create_deep_agent(
    model=model,
    system_prompt='你是一位很有帮助的研究助理。引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。',
)

print('深度代理创建成功！')

深度代理创建成功！


### 测试内置文件系统工具

即使不添加任何自定义工具，我们的代理也已经具备文件系统功能：

In [4]:
# 要求代理写入和读取文件
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '编写一个名为notes.md 的文件，其中包含文本“Hello from Deep Agents!”然后读回以确认。',
            }
        ]
    }
)

# 打印最终回复
print('智能体响应:', result["messages"][-1].content)

智能体响应: 文件 `notes.md` 已创建并确认，内容为：

```
Hello from Deep Agents!
```

写入和读取均已完成。


In [5]:
# 显示虚拟文件系统内容（存储在 result["files"] 中）
print("\n" + "=" * 50)
print('📁 虚拟文件系统（在内存中，而不是在磁盘上！）')
print("=" * 50)
for path, file_data in result.get("files", {}).items():
    print(f"\n  Path: '{path}'")
    print("  " + "-" * 38)
    # file_data 包含“内容”（行列表）、“created_at”、“modified_at”
    if isinstance(file_data, dict) and "content" in file_data:
        content = "\n".join(file_data["content"])
        for line in content.split("\n"):
            print(f"  | {line}")
    else:
        print(f"  | {file_data}")


📁 虚拟文件系统（在内存中，而不是在磁盘上！）

  Path: '/notes.md'
  --------------------------------------
  | H
  | e
  | l
  | l
  | o
  |  
  | f
  | r
  | o
  | m
  |  
  | D
  | e
  | e
  | p
  |  
  | A
  | g
  | e
  | n
  | t
  | s
  | !


### 要点：
- `create_deep_agent()` 免费为您提供文件系统+规划功能
- 无需定义基本文件操作 - 它们是内置的
- 文件存储在代理状态（我们将在第 3 部分中了解更多信息）

## 第 2 部分：添加自定义工具

虽然内置工具很强大，但我们需要**自定义工具**来构建研究代理。让我们添加网络搜索和战略思维功能。

### 使用 Tavily 创建 Web 搜索工具

In [6]:
from langchain_core.tools import tool
from tavily import TavilyClient

# 初始化 Tavily 客户端
tavily_client = TavilyClient()


@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """在网络上搜索有关给定查询的信息。

    Args:
        query: 要执行的搜索查询
    """
    search_results = tavily_client.search(query, max_results=3, topic="general")

    result_texts = []
    for result in search_results.get("results", []):
        url = result["url"]
        title = result["title"]
        content = result.get("content", '无可用内容')
        result_text = f"## {title}\n**URL:** {url}\n\n{content}\n\n---\n"
        result_texts.append(result_text)

    return (
        f"Found {len(result_texts)} result(s) for '{query}':\n\n{''.join(result_texts)}"
    )


print('tavilly_search 工具已创建！')

tavilly_search 工具已创建！


### 将自定义工具添加到我们的代理中

In [7]:
# 使用我们的自定义工具重新创建代理
agent = create_deep_agent(
    model=model,
    tools=[tavily_search],  # 添加我们的自定义工具
    system_prompt="""你是一位很有帮助的研究助理。
    
使用 tavilly_search 在网络上查找信息。
引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。""",
)

print('代理更新了自定义工具！')

代理更新了自定义工具！


In [8]:
# 测试搜索能力
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": '搜索有关 LangGraph 的信息并总结您找到的内容。'}
        ]
    }
)

print(result["messages"][-1].content)

## LangGraph 总结

LangGraph 是由 **LangChain 公司** 于 2024 年 1 月推出的**开源 AI 智能体框架**，专注于构建、部署和管理复杂的生成式 AI 工作流。当前最新版本为 **1.2.6**。

---

### 核心架构

LangGraph 采用**图结构架构**，核心概念包括：

- **节点 (Nodes)** — 处理数据的函数
- **边 (Edges)** — 节点间的连接和路由
- **状态 (State)** — 跨节点的持久化内存/记忆

与传统线性流水线不同，它使用**有向图**来组织工作流，支持条件决策、并行执行和循环结构。

---

### 关键特性

| 特性 | 说明 |
|------|------|
| **持久化执行** | 工作流可在长时间运行中保持状态 |
| **流式输出** | 支持实时流式响应 |
| **人机交互 (Human-in-the-loop)** | 可在关键节点暂停、审批、干预 |
| **检查点与恢复** | 支持故障恢复和时间回溯调试 |
| **多智能体编排** | 支持 supervisor-worker、共识推理等多智能体协作模式 |
| **内存与持久化** | 跨对话和工作流的状态记忆 |

---

### 与 LangChain 的关系

LangGraph 是 LangChain 生态中的**底层编排框架**。LangChain 提供高层 API 用于快速构建智能体应用，而 LangGraph 提供对工作流的**精细控制**。LangChain 智能体本身就构建在 LangGraph 之上以获取持久化执行、流式输出等能力。

---

### 典型应用场景

- 聊天机器人与对话系统
- 多智能体协作系统（如数据分析助手、代码生成）
- 包含人工审批的复杂业务工作流
- 长时间运行的状态化 AI 任务
- 条件决策树和复杂逻辑链路

---

### 采用情况

被众多知名企业采用，包括 **Klarna、Replit、Elastic、Lyft、Vanta、Harvey** 等，用于大规模 AI 应用的生产部署。

---

### 学习资源

LangChain 官方提供 Academy 课程（"Intro to LangGraph"

### 要点：
- 自定义工具扩展了您的代理的能力
- `@tool` 装饰器将函数转换为 LangChain 工具

## 第 3 部分：了解后端

 <img src="../../images/deepAgentBackends.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Backend Architecture"> 

代理的文件实际上去哪里了？ **后端**是可插入存储系统，向代理公开文件系统表面。

### 四种后端类型：

|后端 |存储|坚持|使用案例|
|--------|---------|-------------|---------|
| **状态后端** |InMemory（AgentState）|单线程 |暂存器，中间结果 |
| **文件系统后端** |FileSystem|永久|直接文件访问（谨慎使用！） |
| **商店后端** | LangGraph Store |跨线程 |长期记忆|
| **复合后端** |other |混合|选择性坚持 |

默认情况下，`create_deep_agent()` 使用 **StateBackend** — 文件存储在代理状态中，并在线程结束时消失。

In [9]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

# 添加一个检查点，以便我们可以展示跨回合的持久性
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt='你是一位很有帮助的研究助理。引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。',
    checkpointer=checkpointer,
)

# 创建一个线程
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print(f"Thread ID: {thread_id}")

Thread ID: 019f137a-eb77-71a2-a865-f0192fbe1019


In [10]:
# 在此线程中写入文件
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '编写一个名为 /research_notes.md 的文件，其中包含“我的研究成果放在这里”',
            }
        ]
    },
    config=config,
)

print('Files in state:', list(result.get("files", {}).keys()))

Files in state: ['/research_notes.md']


In [11]:
# 在同一个线程中，文件仍然存在
result = agent.invoke(
    {"messages": [{"role": "user", "content": '读取文件`/research_notes.md`'}]},
    config=config,
)

print(result["messages"][-1].content)

文件内容如下：

```
我的研究成果放在这里
```

内容为单行文本："我的研究成果放在这里"。


In [12]:
# 在新线程中，文件消失了（StateBackend 是短暂的）
new_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": '使用 ls / 列出所有文件'}]},
    config=new_config,
)

print(result["messages"][-1].content)

根目录 `/` 为空，没有任何文件或目录。


### FilesystemBackend - 写入真实磁盘

当您需要代理处理**磁盘上的实际文件**时，请使用 `FilesystemBackend` 。使用 `virtual_mode=True` 时，为了安全起见，路径会在 `root_dir` 下沙箱化。

> ⚠️ **注意**：FilesystemBackend 写入真实文件！使用 `virtual_mode=True` 来防止路径遍历攻击。

In [13]:
from deepagents.backends import FilesystemBackend
import tempfile
import os

# 为我们的沙箱创建一个临时目录
sandbox_dir = tempfile.mkdtemp(prefix="deepagents_sandbox_")
print(f"Sandbox directory: `{sandbox_dir}`")

# 使用 virtual_mode=True 创建 FilesystemBackend （沙盒）
fs_backend = FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True)

# 创建写入真实磁盘的代理（沙盒）
agent_with_fs = create_deep_agent(
    model=model,
    system_prompt='你是一个有用的助手。您写入的文件将转到本地文件系统。引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。',
    backend=fs_backend,
    checkpointer=checkpointer,
)

print('具有文件系统后端的代理已创建！')

Sandbox directory: `/var/folders/c6/4156q1jd2_j940k8w7z13x600000gn/T/deepagents_sandbox_7jv6qoih`
具有文件系统后端的代理已创建！


In [14]:
# 测试：通过代理写文件
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_fs.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '编写一个名为notes.txt 的文件，其中包含“Hello from FilesystemBackend!”',
            }
        ]
    },
    config=config,
)

print('智能体响应:', result["messages"][-1].content)

# 验证文件确实已写入磁盘！
actual_path = os.path.join(sandbox_dir, '笔记.txt')
if os.path.exists(actual_path):
    with open(actual_path, "r") as f:
        print(f"\n✅ File exists on disk at: `{actual_path}`")
        print(f"   Content: {f.read()}")
else:
    print(f"\n❌ File not found at: {actual_path}")

    # 列出沙箱中的所有文件
print(f"\n📁 Files in sandbox ({sandbox_dir}):")
for f in os.listdir(sandbox_dir):
    print(f"   - {f}")

智能体响应: 已创建文件 `notes.txt`，内容为 "Hello from FilesystemBackend!"。

❌ File not found at: /var/folders/c6/4156q1jd2_j940k8w7z13x600000gn/T/deepagents_sandbox_7jv6qoih/笔记.txt

📁 Files in sandbox (/var/folders/c6/4156q1jd2_j940k8w7z13x600000gn/T/deepagents_sandbox_7jv6qoih):
   - notes.txt


### CompositeBackend - 将路径路由到不同的后端

 `CompositeBackend` 允许您将不同的路径路由到不同的后端。这就是实现**混合存储**的方式 - 一些路径是短暂的，其他路径是持久的，其他路径在磁盘上。

 ```
/                 → StateBackend (ephemeral scratch space)
/memories/*       → StoreBackend (persistent across threads)
/workspace/*      → FilesystemBackend (real disk)
```

In [15]:
from deepagents.backends import StateBackend, CompositeBackend

# 为工作区文件创建另一个沙箱
workspace_dir = tempfile.mkdtemp(prefix="deepagents_workspace_")
print(f"Workspace directory: `{workspace_dir}`")

composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # /workspace/* → FilesystemBackend（真实磁盘，沙盒）
        "/workspace/": FilesystemBackend(root_dir=workspace_dir, virtual_mode=True),
    },
)

agent_composite = create_deep_agent(
    model=model,
    system_prompt="""你是一个有用的助手。

储存规则：
- /workspace/* 中的文件保存到真实磁盘（持久）
- 所有其他文件都是短暂的（线程结束时消失）

引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。""",
    backend=composite_backend,
    checkpointer=checkpointer,
)

print('具有 CompositeBackend 的代理已创建！')

Workspace directory: `/var/folders/c6/4156q1jd2_j940k8w7z13x600000gn/T/deepagents_workspace_3ox6fbqw`
具有 CompositeBackend 的代理已创建！


In [ ]:
# 测试：写入两个位置并查看差异
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_composite.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """写入两个文件：
1. `/workspace/persistent.txt` 与“我会活下去！”
2. `/scratch.txt` 与“我是短暂的”

然后列出两个位置。""",
            }
        ]
    },
    config=config,
)

print('智能体响应:', result["messages"][-1].content)

# 检查磁盘上的内容和状态
print("\n" + "=" * 50)
print('📁 磁盘上（workspace_dir）：')
for f in os.listdir(workspace_dir):
    filepath = os.path.join(workspace_dir, f)
    with open(filepath, "r") as file:
        print(f"   {f}: {file.read()}")

print('📦 在虚拟状态下：')
for path, data in result.get("files", {}).items():
    if isinstance(data, dict) and "content" in data:
        content = "\n".join(data["content"])
    else:
        content = str(data)
    print(f"   `{path}`: {content}")

### StoreBackend - 用于 LangSmith 部署

 `StoreBackend` 使用 LangGraph 的 `BaseStore` 进行持久、跨线程存储。这是您在以下情况下将使用的后端：

- 本地运行 `langgraph dev` （自动提供存储）
- 部署到**LangSmith Deployments**（商店由平台管理）

该平台提供持久存储，因此您的代理的 `/memories/*` 文件可以在对话和服务器重新启动时保留下来。

> 💡 **注意**：我们将在第 6 部分（长期记忆）中演示 `StoreBackend`，其中我们将其与 `CompositeBackend` 结合起来以实现选择性持久性。

 ```python
# When deployed to LangSmith Deployment, the store is injected automatically
# This is the pattern you'll use in production:
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend
from langgraph.store.memory import InMemoryStore

agent = create_deep_agent(
    backend=CompositeBackend(
        default=StateBackend(),  # Everything else is ephemeral
        routes={
            # /memories/* persists across threads via the platform's store
            "/memories/": StoreBackend(),
        }
    ),
    store=InMemoryStore()  # Good for local dev; omit for LangSmith Deployment
)
```

In [16]:
# 清理：删除临时目录
import shutil

shutil.rmtree(sandbox_dir, ignore_errors=True)
shutil.rmtree(workspace_dir, ignore_errors=True)
print('✅ 清理临时目录')

✅ 清理临时目录


### 要点：
- **后端**控制代理文件的存储位置/方式
- **StateBackend**（默认）是短暂的 - 文件在线程结束时消失
- **FilesystemBackend** 写入真实磁盘（使用 `virtual_mode=True` 进行沙箱）
- **CompositeBackend** 将不同的路径路由到不同的后端（混合存储）
- **StoreBackend** 用于 LangSmith 部署（我们将在第 6 部分中使用它）

## 第 4 部分：添加研究子代理

 <img src="../../images/deepAgentSubagents.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Subagent Architecture"> 

随着代理完成更多工作，他们的上下文会充满中间工具调用。 **子代理**通过在单独的上下文中隔离工作来解决这个问题。

### 上下文膨胀问题

没有子代理：
 ```
User: Research AI agents
→ search("AI agents overview") → 5000 tokens of results
→ think("Found overview...") → 200 tokens
→ search("AI agent frameworks") → 5000 tokens of results  
→ think("Found frameworks...") → 200 tokens
→ ... context bloats with every search
``` 

与子代理：
 ```
User: Research AI agents
→ task("research-agent", "Research AI agents")
→ [Subagent does all searches in isolated context]
→ Returns: "Summary: AI agents are..." (clean, compressed result)
```

In [17]:
from datetime import datetime

# 获取研究人员的当前日期
current_date = datetime.now().strftime("%Y-%m-%d")

# 定义研究员子代理
RESEARCHER_INSTRUCTIONS = f"""你是正在执行研究任务的研究助手。今天的日期是 {current_date}。

<Task>
使用工具收集研究主题相关的信息。
</Task>

<Hard Limits>
- 简单查询：最多使用 2-3 次搜索工具调用
- 复杂查询：最多使用 5 次搜索工具调用
</Hard Limits>

<Output Format>
请按以下结构组织你的发现：
- 清晰的标题
- 行内引用 [1]、[2]、[3]
- 末尾的来源部分
</Output Format>

引用文件路径时，请使用反引号格式，例如 `path/file.md`，不要使用 Markdown 链接。
"""

research_subagent = {
    "name": "research-agent",
    "description": '委托研究任务。一次给出一个主题。',
    "system_prompt": RESEARCHER_INSTRUCTIONS,
    "tools": [tavily_search],
}

print('研究子代理已定义！')
print(f"  Name: {research_subagent['name']}")
print(f"  Tools: {[t.name for t in research_subagent['tools']]}")

研究子代理已定义！
  Name: research-agent
  Tools: ['tavily_search']


In [18]:
# 定义协调器指令
ORCHESTRATOR_INSTRUCTIONS = """您是一名研究协调员。

当被要求研究一个主题时：
1. 使用 write_todos 来计划你的研究任务
2. 使用task()工具将研究委托给研究代理子代理
3. 切勿直接搜索 - 始终委托给研究代理
4. 综合结果并将报告写入 /final_report.md

研究代理将处理所有网络搜索并返回总结结果。

引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。"""

# 创建具有子代理的代理
agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=ORCHESTRATOR_INSTRUCTIONS,
    subagents=[research_subagent],  # 添加我们的研究子代理
    checkpointer=checkpointer,
)

print('已创建带有子代理的代理！')

已创建带有子代理的代理！


In [ ]:
# 测试委托
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": '简单研究一下本周有关人工智能代理的有趣新闻'}
        ]
    },
    config=config,
)

print(result["messages"][-1].content[:2000] + "...")

### 要点：
- 子代理将工作隔离在单独的上下文中
- 主代理只能看到最终结果，看不到中间搜索
- 这使主要代理的上下文保持干净和集中

## 第 5 部分：中间件深入探讨

Deep Agents 使用**模块化中间件架构**。当您调用 `create_deep_agent()` 时，会自动附加几个中间件组件。
 <img src="../../images/deepAgentMiddleware.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Middleware"> 

### 始终在线的中间件：

|中间件|提供的工具 |目的|
|------------|--------------|----------|
| **TodoList中间件** |  `write_todos` |任务规划和跟踪|
| **文件系统中间件** |  `ls` 、 `read_file` 、 `write_file` 、 `edit_file` 、 `glob` 、 `grep` |文件操作 + **大型工具结果驱逐** |
| **子代理中间件** |  `task` |将工作委派给子代理 |
| **摘要中间件** | *（无）* |以约 85% 的上下文容量压缩对话历史记录 |
| **PatchToolCallsMiddleware** | *（无）* |修复消息历史记录中悬空工具调用 |

### 条件中间件（配置时添加）：

|中间件|触发|目的|
|------------|---------|----------|
| **内存中间件** |  `memory=["/AGENTS.md"]` |从 AGENTS.md 文件加载持久上下文 |
| **技能中间件** |  `skills=["/skills/"]` |捆绑功能的逐步公开|
| **HumanInTheLoop中间件** |  `interrupt_on={...}` |敏感操作的人工批准 |

### 内置上下文管理策略深度代理自动管理上下文以保持在模型的令牌限制内。共有三种策略，均可通过 `tool_token_limit_before_evict` 和模型的 `max_input_tokens` 进行配置：<风格>
.ctx-grid { 显示：网格；网格模板列：重复（自动调整，minmax（500px，1fr））；间隙：10px；边距：10px 0 10px 0； }
.ctx-card img { width: 1000px; border-radius: 8px; margin-bottom: 8px; } 
.ctx-卡 p { font-size: 0.92em; line-height: 1.5; margin: 0; } 
.ctx-卡强{ display: block; margin-bottom: 4px; } 
</风格>
<div class="ctx-grid">
<div class="ctx-card">
 <img src="../../images/Offloading%20Inputs%20LangChain.png" alt="Offloading tool inputs"> 
<p><strong>卸载大量输入</strong>文件写入/编辑工具调用会将完整内容保留在对话历史记录中。由于它已经在磁盘上，因此它是多余的。在上下文容量约为 85% 时，深度代理会截断这些较旧的工具调用，并将其替换为指向文件的指针。</p>
</div>
<div class="ctx-card">
 <img src="../../images/Offloading%20Results%20LangChain.png" alt="Offloading tool results"> 
<p><strong>卸载大型结果</strong>当工具结果超过约 20k 令牌时，深度代理会将其保存到后端，并将其与文件路径引用 + 10 行预览进行交换。代理可以根据需要重新读取或搜索完整内容。</p>
</div>
<div class="ctx-card">
 <img src="../../images/LangChain%20Summarization.png" alt="Conversation summarization"> 
<p><strong>对话摘要</strong>当上下文达到 <code>max_input_tokens</code> 的约 85% 并且没有任何内容可以卸载时，代理会总结历史记录。完整消息保存在磁盘上的 <code>/conversation_history/</code> 中；结构化摘要会在工作记忆中取代它们。</p>
</div>
</div>

In [19]:
# 我们的代理已经拥有所有三个中间件！
# 让我们看看实际的规划能力

config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '制定研究机器学习框架的计划。使用 write_todos。',
            }
        ]
    },
    config=config,
)

print(result["messages"][-1].content)

研究完成！综合报告已写入 **`/final_report.md`**。

### 研究执行概要

通过 4 个并行研究代理，对 **6 个主流机器学习框架** 进行了调研：

| 框架 | GitHub Stars | 最新版本 | 核心定位 |
|------|:-----------:|:--------:|---------|
| **PyTorch** | 80,000+ | v2.10.0 | 研究 & 深度学习首选 |
| **TensorFlow** | 185,000+ | v2.18.x | 企业生产部署 |
| **scikit-learn** | 60,000+ | v1.6.x | 传统机器学习标准 |
| **Keras** | 62,000+ | v3.x | 三后端高级 API |
| **JAX** | 30,000+ | v0.6.x | 高性能科学计算 |
| **MXNet** | 21,000+ | 维护模式 | 已被取代 |

### 关键发现

- **研究领域**：PyTorch 以 **55–85%** 的论文占比绝对主导
- **生产部署**：TensorFlow 在移动端（LiteRT）和 TPU 场景仍有优势
- **传统 ML**：scikit-learn 依然是表格数据和经典算法的标准
- **新趋势**：Keras v3 支持三后端切换，JAX 在高性能计算场景崛起

报告包含完整的 **框架对比总表** 和 **按场景的选型建议**，详见 `final_report.md`。


In [20]:
# 检查状态中的待办事项
if "todos" in result:
    print('📋 代理待办事项清单：')
    for todo in result["todos"]:
        status_map = {"completed": "✅", "in_progress": "🔄", "pending": "⬚"}
        status = todo.get("status", "pending")
        icon = status_map.get(status, "⬚")
        content = todo.get("content", str(todo))
        print(f"  {icon} {content}")
else:
    print('状态中没有待办事项（代理可能使用了不同的方法）')

📋 代理待办事项清单：
  ✅ 研究 TensorFlow 框架 - 最新版本、特点、生态系统
  ✅ 研究 PyTorch 框架 - 最新版本、特点、生态系统
  ✅ 研究 scikit-learn 框架 - 最新版本、特点、适用场景
  ✅ 研究 Keras / JAX / MXNet 等其他主流框架
  ✅ 综合研究结果，撰写最终报告并写入 /final_report.md


### 自定义中间件：记录工具调用

除了内置中间件之外，您还可以编写自己的中间件。最简单的方法是使用 **基于装饰器的中间件** 使用像 `@wrap_tool_call` 这样的钩子，它包围每个工具调用。

这对于可观察性很有用——实时准确地看到代理正在做什么。

可用的钩子：

|钩|当它运行时 |风格|
|------|-------------|--------|
|  `@before_agent` |代理启动之前（一次）|节点|
|  `@before_model` |在每次LLM通话之前|节点|
|  `@after_model` |每次LLM回复后|节点|
|  `@after_agent` |代理完成后（一次）|节点|
|  `@wrap_model_call` |围绕每个LLM通话|包裹 |
|  `@wrap_tool_call` |围绕每个工具调用 |包裹 |

让我们创建一个简单的 `@wrap_tool_call` 中间件来记录代理使用的每个工具：

In [21]:
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def log_tool_calls(request, handler):
    """记录代理进行的每个工具调用。"""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"🔧 [Tool Call] {tool_name}")
    print(f"   Args: {tool_args}")

    result = handler(request)

    print(f"✅ [Tool Done] {tool_name}\n")
    return result

    # 使用我们的自定义中间件创建代理


agent_with_logging = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt='你是一位很有帮助的研究助理。引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。',
    middleware=[log_tool_calls],
    checkpointer=MemorySaver(),
)

In [ ]:
# 运行代理——观察输出中的工具调用日志！
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_logging.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '什么是 LangGraph？在您的文件系统中创建一个简短的摘要。',
            }
        ]
    },
    config=config,
)

print('--- 代理响应 ---')
print(result["messages"][-1].content)

### 要点：
- 中间件 = 可插入的功能模块，封装代理的模型调用和工具调用
- `create_deep_agent()` 自动添加 TodoList、Filesystem、SubAgent、Summarization 和 PatchToolCalls 中间件
- **内置上下文管理**：大型工具结果被逐出，对话历史记录被总结
- 编写自定义中间件非常简单 - 使用像 `@wrap_tool_call` 这样的装饰器钩子来实现快速的单一用途中间件，或者使用基于类的 `AgentMiddleware` 来实现更复杂的情况
- 通过`middleware=[...]`传递自定义中间件，它与所有内置中间件组合

## 第 6 部分：人机交互

 <img src="../../images/deepAgentHITL.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Human-in-the-Loop Flow"> 

对于敏感操作，您可能希望有人在执行操作之前批准这些操作。 Deep Agents 通过 `HumanInTheLoopMiddleware` 支持内置**中断**。

### 内置决策类型：
- **批准** — 使用建议的参数执行
- **编辑** — 执行前修改参数
- **拒绝** — 完全跳过工具调用

此外，您还可以自定义每个工具可用的决策，例如：
 ```python
interrupt_on = {
    # Sensitive operations: allow all options
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},

    # Moderate risk: approval or rejection only
    "write_file": {"allowed_decisions": ["approve", "reject"]},

    # Must approve (no rejection allowed)
    "critical_operation": {"allowed_decisions": ["approve"]},
}
``` 

#### 对于此演示，我们使用默认的中断决策类型：

In [22]:
# 创建具有文件写入中断的代理
agent_with_hitl = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt='你是一位很有帮助的研究助理。引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。',
    subagents=[research_subagent],
    checkpointer=checkpointer,
    interrupt_on={
        "write_file": True,  # 写入文件之前中断
        "edit_file": True,  # 编辑文件之前中断
    },
)

print('已创建具有 HITL 的代理！')

已创建具有 HITL 的代理！


#### 让我们来尝试一下吧！

In [23]:
from langgraph.types import Command

config = {"configurable": {"thread_id": str(uuid7())}}

# 当代理尝试写入文件时，这将触发中断
result = agent_with_hitl.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '编写一个名为 /test.md 的文件，其中包含“Hello World”',
            }
        ]
    },
    config=config,
)

# 检查我们是否遇到中断
if result.get("__interrupt__"):
    print('🛑 中断被触发！')
    interrupt_value = result["__interrupt__"][0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = interrupt_value["review_configs"]

    for action, review in zip(action_requests, review_configs):
        print(f"  Tool: {action['name']}")
        print(f"  Args: {action['args']}")
        print(f"  Allowed decisions: {review['allowed_decisions']}")
else:
    print('无中断（文件已写入）')
    print(result["messages"][-1].content)

🛑 中断被触发！
  Tool: write_file
  Args: {'file_path': '/test.md', 'content': 'Hello World'}
  Allowed decisions: ['approve', 'edit', 'reject', 'respond']


#### 经批准后恢复

In [24]:
if result.get("__interrupt__"):
    # 批准写入操作
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}), config=config
    )
    print('✅ 获批准恢复！')
    print(result["messages"][-1].content)

✅ 获批准恢复！
文件 `/test.md` 已创建，内容为 "Hello World"。


### 要点：
- HITL 增加了对危险操作的人工监督
- 配置哪些工具需要 `interrupt_on` 批准 
- HITL 工作需要检查点

## 第 7 部分：长期记忆

 <img src="../../images/deepAgentMemories.png" style="width: auto; height:500px; border-radius: 8px;" alt="Long-Term Memory"> 

到目前为止，当线程结束时文件就会消失。 **长期内存** 使用 `CompositeBackend` 将某些路径路由到持久存储。

### 路径路由：
- `/memories/*` → **StoreBackend**（跨线程持久化）
- 其他一切 → **StateBackend** （短暂的）

In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend

store = InMemoryStore()

composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # /memories/ 下的文件转到持久存储
        "/memories/": StoreBackend(),
    },
)

print('复合后端已创建！')

In [ ]:
# 创建具有长期记忆的代理
agent_with_memory = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="""你是一位有用的研究助理，具有长期记忆力。
    
重要提示：将重要笔记保存到 /memories/ 中，以便它们在对话中持续存在。
例如：/memories/research_notes.md

对话结束时，常规文件（不在 /memories/ 中）将消失。

当被问到你记得或知道什么时，总是使用 ls 和 read_file 首先检查 /memories/。
不要仅仅依赖于对话上下文——记忆会跨线程持续存在，但对话不会。

引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。""",
    subagents=[research_subagent],
    checkpointer=checkpointer,
    backend=composite_backend,
    store=store,  # Store是在这里传递的，不是直接传到后端
)

print('具有长期记忆的特工已创建！')

In [ ]:
# 主题 1：将某些内容保存到长期记忆中
thread1_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_memory.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '将“重要研究发现：人工智能代理正在迅速发展”保存到 `/memories/findings.md`',
            }
        ]
    },
    config=thread1_config,
)

print('Thread 1:', result["messages"][-1].content)

In [ ]:
# 线程2：从不同的线程访问内存
thread2_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_memory.invoke(
    {
        "messages": [
            {"role": "user", "content": '读一下文件 `/memories/findings.md` 中的内容'}
        ]
    },
    config=thread2_config,
)

print('Thread 2:', result["messages"][-1].content)

### 要点：
- `CompositeBackend` 将不同的路径路由到不同的存储后端
- `/memories/*` 跨线程持续存在
- 其他文件保持短暂（单线程）

### 高级记忆模式：语义、情景和程序

认知科学研究（以及 [CoALA 论文]( https://arxiv.org/abs/2309.02427)) 确定了三种类型的长期记忆，它们自然地映射到智能体如何存储信息：

|内存类型 |它存储什么 |人类的例子|代理示例 |
|------------|----------------|----------------|---------------|
| **语义** |事实与知识| “巴黎是法国的首都”|用户偏好、项目背景 |
| **情景** |过往经历 | “上周二我去徒步旅行”|过去的研究会议、互动日志 |
| **程序** |说明和规则 |如何骑自行车 |编码标准、报告格式规则 |

我们可以使用 `CompositeBackend` 和多个路由将它们映射到 **文件系统路径**：

 ```
/memories/semantic/       -> Facts: user_preferences.md, project_context.md
/memories/episodic/       -> Experiences: 2025-02-17_research.md
/memories/procedural/     -> Rules: coding_standards.md, report_format.md
/                         -> Ephemeral scratch space (StateBackend)
``` 

由于 CompositeBackend 使用 **最长前缀匹配**，因此 `/memories/semantic/` 优先于更广泛的 `/memories/` 路由。

In [26]:
# 创建一个包含每种内存类型路由的 CompositeBackend

memory_store = InMemoryStore()

advanced_composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/semantic/": StoreBackend(
            namespace=lambda rt: ("memories", "semantic")
        ),
        "/memories/episodic/": StoreBackend(
            namespace=lambda rt: ("memories", "episodic")
        ),
        "/memories/procedural/": StoreBackend(
            namespace=lambda rt: ("memories", "procedural")
        ),
    },
)

print('使用 3 条内存类型路由创建的高级复合后端！')

NameError: name 'InMemoryStore' is not defined

In [ ]:
# 创建一个理解三种内存类型的代理
memory_agent = create_deep_agent(
    model=model,
    system_prompt="""你是一个有结构化长期记忆的有用助手。

你的记忆分为三种类型：
- /memories/semantic/ -> 事实和知识（用户偏好、项目详细信息）
- /memories/episodic/ -> 过去的经历（会话日志、交互摘要）  
- /memories/procedural/ -> 说明和规则（如何格式化报告、编码标准）

当要求记住某些内容时，将其保存到适当的内存类型。
常规文件（不在 /memories/ 中）是短暂的，并且在对话后消失。

当被问及您记得或了解有关该用户的信息时，请始终使用 ls 和 read_file 来检查
首先是内存目录。不要仅依赖对话上下文。

引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。""",
    checkpointer=checkpointer,
    backend=advanced_composite_backend,
    store=memory_store,
)

# 线程 1：写入所有三种内存类型
config_t1 = {"configurable": {"thread_id": str(uuid7())}}

result = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """请将以下内容保存到适当的内存类型中：
1. 相比 JavaScript，我更喜欢 Python（这是我的事实）
2.上一期我们研究了LangGraph，发现它很有用（这是过去的经验）
3. 在研究报告中始终使用内联引用[1]、[2]（这是一条规则）""",
            }
        ]
    },
    config=config_t1,
)

print(result["messages"][-1].content)

In [ ]:
# 线程 2：验证内存在线程之间持续存在
config_t2 = {"configurable": {"thread_id": str(uuid7())}}

result = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '你还记得我什么？检查所有记忆类型：语义记忆、情景记忆和程序记忆。',
            }
        ]
    },
    config=config_t2,
)

print('来自新线程：')
print(result["messages"][-1].content)

### 命名空间范围：每用户与全局内存

默认情况下，`StoreBackend` 将文件存储在命名空间 `(assistant_id, "filesystem")` 中，这意味着一个助手的所有用户共享相同的内存。但是如果您想要**每用户隔离**怎么办？

 `StoreBackend` 接受 `namespace` 参数——一个接收 `BackendContext` 并返回命名空间元组的可调用函数。这控制数据隔离：

|范围 |命名空间|谁能看到它 |
|------|----------|----------------|
| **默认** |  `(assistant_id, "filesystem")` |一位助手的所有用户|
| **每用户** |  `("user", user_id, "filesystem")` |仅该特定用户|
| **全球** |  `("shared", "filesystem")` |所有助手中的所有用户|

这使您可以在以下位置构建代理：
- `/memories/user/` 存储每个用户的私人偏好（由 `user_id` 隔离）
- `/memories/shared/` 存储每个人都可见的团队指南

In [25]:
scoped_store = InMemoryStore()

scoped_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # 每个用户的记忆：命名空间 lambda 接收运行时，
        # 所以 rt.context 获取当前调用的 user_id
        "/memories/user/": StoreBackend(
            namespace=lambda rt: (
                "user",
                getattr(rt.context, "user_id", "default"),
                "filesystem",
            ),
        ),
        # 共享内存：所有用户使用相同的命名空间
        "/memories/shared/": StoreBackend(
            namespace=lambda rt: ("shared", "filesystem"),
        ),
    },
)


scoped_agent = create_deep_agent(
    model=model,
    system_prompt="""你是一个有范围记忆的有用助手。

内存范围：
- /memories/user/ -> 当前用户私有（只有他们可以看到）
- /memories/shared/ -> 在所有用户之间共享（每个人都可以看到它）

将个人偏好保存到 /memories/user/ 并将团队指南保存到 /memories/shared/。

当被问到你记得什么时，总是使用 ls 和 read_file 首先检查内存目录。

引用文件路径时，请使用反引号格式（例如 `path/file.md`）而不是 Markdown 链接。""",
    checkpointer=checkpointer,
    backend=scoped_backend,
    store=scoped_store,
)

print('使用每个用户和共享内存创建作用域代理！')

NameError: name 'InMemoryStore' is not defined

In [ ]:
# 用户 A 写入私有内存和共享内存
config_alice = {"configurable": {"thread_id": str(uuid7()), "user_id": "alice"}}

result = scoped_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """保存这两件事：
1.我的私人记忆（/memories/user/）：“Alice更喜欢深色模式和Python”
2. 共享内存（/memories/shared/）：“团队指南：始终编写单元测试”""",
            }
        ]
    },
    config=config_alice,
)

print('爱丽丝写道：', result["messages"][-1].content)

In [ ]:
# 用户 B 尝试读取两者 - 他们能看到 Alice 的私人记忆吗？
config_bob = {"configurable": {"thread_id": str(uuid7()), "user_id": "bob"}}

result = scoped_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '列出 `/memories/user/` 和 `/memories/shared/` 中的所有文件以查看可以访问的内容。',
            }
        ]
    },
    config=config_bob,
)

print('鲍勃看到：', result["messages"][-1].content)
print("\n" + "=" * 50)
print('鲍勃可以看到共享的准则，但看不到爱丽丝的私人偏好！')

### 要点：
- **三种内存类型**（语义、情景、程序）通过 `CompositeBackend` 路由自然映射到文件系统路径
- ** `StoreBackend` 上的 `namespace` ** 控制数据隔离 - 每个用户、每个助理或全局
- 较长的路由前缀在 `CompositeBackend` 中优先（例如， `/memories/semantic/` 优先于 `/memories/` ）
- 通过配置将 `user_id` 传递到每个用户的范围内存：`config={"configurable": {"user_id": "alice"}}`

## 第 8 部分：AGENTS.md 和技能

到目前为止，我们一直在代码中直接编写 `system_prompt` 字符串。 Deep Agents 提供了两种基于文件的替代方案，它们功能更强大并且遵循最佳实践：

### AGENTS.md：持久身份和说明

 `AGENTS.md` 文件提供持久上下文，**始终通过 `memory=` 参数加载**到系统提示符中。您可以在此处放置代理的身份、工作流程规则和首选项。 AGENTS.md 文件的另一个主要优点是它们采用简单的 Markdown 格式，任何人都可以轻松编辑。 

强大的部分：**代理可以读取和编辑自己的 AGENTS.md 文件**。这意味着代理可以根据反馈更新自己的指令，本质上是一个可自我修改的系统提示。

> **权衡**：AGENTS.md 对于开发和内部代理来说非常强大——代理可以通过编辑自己的指令来学习和自我改进。但在**生产**中，您需要将关键指令移至 `system_prompt` （代理无法编辑），以防止代理意外破坏其自身行为。认为 AGENTS.md 对于“元学习”阶段非常有用。

### 技能：按需能力技能是可重用的功能，与 frontmatter 元数据捆绑为 `SKILL.md` 文件。他们使用**渐进式披露**：
1. 启动时，仅加载技能**名称+描述**（frontmatter）——仅几个标记
2. 当代理确定某个技能与当前任务相关时，它会读取**完整的SKILL.md**内容
3. 这可以使提示保持干净，直到真正需要该技能为止

### 比较

|方法|加载时间 |由代理编辑 |最适合 |
|----------|-------------|--------------------|----------|
|  `system_prompt` |永远 |没有 |核心身份，不可改变的规则|
|  `AGENTS.md`（内存）|永远 |是的 |工作流程、偏好、可学习的规则 |
|  `SKILL.md`（技能）|按需 |否（只读）|特定于任务的说明、模板 |

In [ ]:
# 定义我们的AGENTS.md——代理的身份和指令
agents_md_content = """# 研究助理

您是一名专家研究助理，可以搜索网络、综合研究结果并生成精美的报告和内容。

## 工作流程

1. **计划** -- 使用 `write_todos` 将任务分解为步骤
2. **研究** -- 使用 tavily_search 搜索信息
3. **反思**——每次搜索后反思并分析结果
4. **综合**——将调查结果合并成综合报告
5. **写入** -- 将最终报告保存到`/final_report.md` 
6. **记住** -- 将关键要点保存到 `/memories/research_notes.md` 

## 规则

- 最多使用 2-3 次搜索
- 合并引用——每个唯一的 URL 都有一个数字 [1]、[2]、[3]
- 以来源部分结束报告
- 当要求创建特定内容格式时检查相关技能"""

print('AGENTS.md 已定义！')
print(agents_md_content)

### 现在让我们定义代理的技能

In [ ]:
# 定义 LinkedIn 发帖技能
linkedin_skill_content = """---
名称：linkedin-post
描述：根据研究结果或给定主题撰写 LinkedIn 帖子。当被要求创建 LinkedIn 内容、专业帖子或思想领导力文章时，请使用此技能。
---

# LinkedIn 发帖技巧

## 格式
- **挂钩**：以吸引注意力的大胆开场白开头（出现在“查看更多”剪辑之前）
- **正文**：3-5 个短段落，每个段落 1-2 个句子
- 在段落之间使用换行符以提高可读性
- 每段包含 1-2 个相关表情符号（不要过多）
- 以号召性用语或问题结束以提高参与度
- 在底部添加 3-5 个相关主题标签

## 语气
- 专业但健谈
- 分享见解，而不仅仅是信息
- 在适当的情况下使用“我”的陈述和个人观点

## 长度
- 理想：150-300字
- LinkedIn 在约 210 个字符后截断，因此第一行必须吸引读者"""

print('LinkedIn 发帖技能定义！')
print(f"Skill name: linkedin-post")
print(f"Length: {len(linkedin_skill_content)} chars")

In [ ]:
# 定义 Twitter/X 帖子技能
twitter_skill_content = """---
名称：推特帖子
描述：根据研究结果或给定主题撰写 Twitter/X 帖子或主题。当被要求创建推文、X 帖子或社交媒体帖子时，请使用此技能。
---

# Twitter/X 发帖技巧

## 单条推文格式
- 最多 280 个字符
- 以最引人注目的观点开头
- 尽可能使用数字或数据
- 最多 1-2 个主题标签（可选）

## 线程格式（适用于较长的内容）
- 推文 1：挂钩 + 预览（例如，“X 上的线程：”）
- 推文 2-N：每条推文一个想法，编号为 (1/, 2/, 3/)
- 最后推文：摘要 + 号召性用语
- 4-8 条推文是最佳选择

## 语气
- 简洁有力
- 自以为是的总结比中立的总结表现更好
- 使用简单的语言——不使用公司语言"""

print('Twitter/X 帖子技能定义！')
print(f"Skill name: twitter-post")
print(f"Length: {len(twitter_skill_content)} chars")

### 把它们放在一起（AGENTS.md + 技能）

In [ ]:
# 创建使用 AGENTS.md + 技能的代理
# 在笔记本中，我们使用 create_file_data() 通过 `files` 状态键播种文件
# 在生产中（langgraph dev / Studio），您将使用 FilesystemBackend 并从磁盘加载
from deepagents.backends.utils import create_file_data

skill_agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt='你是一位专家研究助理。',
    memory=['/AGENTS.md'],  # 始终加载到系统提示符中
    skills=["/skills/"],  # 通过渐进式披露按需加载
    checkpointer=checkpointer,
    backend=composite_backend,
    store=store,
)

# 将 AGENTS.md 和技能文件播种到虚拟文件系统中
skill_files = {
    '/AGENTS.md': create_file_data(agents_md_content),
    '/skills/linkedin-post/SKILL.md': create_file_data(linkedin_skill_content),
    '/skills/twitter-post/SKILL.md': create_file_data(twitter_skill_content),
}

print('使用 AGENTS.md + 2 项技能创建的代理！')
print(f"  Memory: `/AGENTS.md` (always loaded)")
print(f"  Skills: linkedin-post, twitter-post (loaded on demand)")

In [ ]:
# 测试：研究一个主题，然后请求 LinkedIn 帖子
# 代理将首先进行研究（不需要技能），然后按需加载 linkedin-post 技能
config = {"configurable": {"thread_id": str(uuid7())}}

result = skill_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '研究什么是人工智能代理，然后在 LinkedIn 上发表一篇文章介绍您的发现。',
            }
        ],
        "files": skill_files,
    },
    config=config,
)

print(result["messages"][-1].content)

In [ ]:
# 测试：现在在同一对话中请求一条推文线程
# twitter-post 技能将按需加载
result = skill_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": '现在写一个关于同一主题的 Twitter/X 线程。'}
        ]
    },
    config=config,
)

print(result["messages"][-1].content)

### 要点：
- **AGENTS.md** 替换硬编码的 `system_prompt` 字符串 - 通过 `memory=` 加载，可由代理编辑
- **技能**通过渐进式披露提供按需功能——仅在相关时加载
- 在笔记本中，通过 `files=` 和 `create_file_data()` 种子文件；在生产中，使用 `FilesystemBackend` 从磁盘加载
- 代理可以通过编辑自己的 AGENTS.md 进行自我改进——对于开发来说非常强大，但在生产中锁定 `system_prompt` 中的关键规则

## 第 9 部分：完整的研究代理

让我们回顾一下我们构建的内容！从基本的 `create_deep_agent()` 调用开始，我们逐步添加：

 ```
Part 1: create_deep_agent(model)                    → Basic filesystem agent
Part 2: + tools=[tavily_search]                     → Can search web
Part 3: (understand backends)                       → Same agent, understood storage
Part 4: + subagents=[research_agent]                → Can delegate research
Part 5: + middleware=[log_tool_calls]               → Log tool calls as they happen
Part 6: + interrupt_on={...}, checkpointer          → Human oversight
Part 7: + backend (CompositeBackend)                → Long-term memory
Part 8: + memory (AGENTS.md) + skills (SKILL.md)    → Self-modifiable identity + on-demand capabilities
``` 

现在让我们使用 AGENTS.md + 技能模式构建最终代理：

In [ ]:
# 创建我们的最终研究代理——使用 AGENTS.md + 技能（就像 Studio 代理一样）
final_agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt='你是一位专家研究助理。',
    middleware=[log_tool_calls],
    skills=["/skills/"],  # LinkedIn + Twitter 技能按需加载
    memory=['/代理.md'],  # 来自 AGENTS.md 的身份 + 工作流程
    checkpointer=checkpointer,
    backend=composite_backend,
    store=store,
)

# 将文件播种到虚拟文件系统中
final_agent_files = {
    '/代理.md': create_file_data(agents_md_content),
    '/技能/linkedin-post/SKILL.md': create_file_data(linkedin_skill_content),
    '/技能/twitter-post/SKILL.md': create_file_data(twitter_skill_content),
}

print('使用 AGENTS.md + 技能创建的最终研究代理！')

In [ ]:
# 运行完整的研究工作流程——研究 + LinkedIn 帖子（锻炼技能！）
config = {"configurable": {"thread_id": str(uuid7())}}

print('开始研究工作流程...')

result = final_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": '研究什么是 LangChain Deep Agent，写一份简短的报告，然后在 LinkedIn 上发表一篇关于您的发现的帖子。',
            }
        ],
        "files": final_agent_files,
    },
    config=config,
)

print(result["messages"][-1].content[:2000])

In [ ]:
# 显示虚拟文件系统 - 代理是否写入文件？
print("\n" + "=" * 60)
print('📁 虚拟文件系统')
print("=" * 60)
for path, file_data in result.get("files", {}).items():
    if isinstance(file_data, dict) and "content" in file_data:
        content = "\n".join(file_data["content"])
    else:
        content = str(file_data)
    print(f"\n📄 '{path}' ({len(content)} chars)")
    print("-" * 40)
    print(content[:500] + ("..." if len(content) > 500 else ""))

### 要点：
- Deep Agents 可以逐步构建复杂的代理
- 每个功能（工具、子代理、HITL、内存）都是可组合的
- 框架负责编排，您专注于功能

## 第 10 部分：后续步骤

恭喜！你已经使用 Deep Agents 构建了一个完整的研究智能体。这里是你学到的内容：

| 概念 | 作用 |
|---------|-------------|
| **智能体框架** | 预构建工具 + 上下文管理 |
| **自定义工具** | 扩展能力（例如搜索） |
| **后端** | 控制文件存储：临时、磁盘、持久化或混合 |
| **子智能体** | 为复杂任务隔离上下文 |
| **中间件** | 可插拔的功能模块（文件系统、摘要、待办事项等） |
| **人在环** | 为敏感操作提供安全关卡 |
| **长期记忆** | 通过路径路由和命名空间范围实现持久化存储 |
| **AGENTS.md** | 基于文件的智能体身份说明：始终加载，也可由智能体编辑 |
| **技能** | 通过渐进式披露按需加载能力（SKILL.md 文件） |

### 资源

- [Deep Agents 文档](https://github.com/langchain-ai/deepagents)
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)
- [LangChain Academy](https://academy.langchain.com/)
- [LangChain vs LangGraph vs Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview)

### 接下来可以做什么？

1. **在 Studio 中运行** - 查看 `agents/deep_agent`，那里有带 AGENTS.md 和磁盘技能的生产就绪版本
2. **部署** - 使用 LangSmith Deployment 部署到生产环境（`langgraph dev` 或 LangSmith Deployments）
3. **添加技能** - 为领域专属能力创建自己的 SKILL.md 文件
4. **自定义记忆** - 按用户和共享记忆使用命名空间范围
5. **扩展系统** - 使用专门的子智能体构建多智能体系统

<br>
<br>
---
<br>

**构建愉快！**
